# Module 2: Sales Forecasting with Python
### Python Programming & AI — Hochschule für Wirtschaft und Umwelt (HfWU)

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Load and explore** time series sales data using `pandas`
2. **Engineer date and lag features** from raw timestamps
3. **Split time series data correctly** — no random splits allowed
4. **Train a Linear Regression model** to predict future sales
5. **Visualize predictions** against actual sales with `plotly`
6. **Package the workflow** into a reusable sklearn Pipeline

---

## The Big Idea

> *"Can we predict next month's revenue from past data?"*

Every business needs to answer this question:
- How much stock should we order next month?
- How many staff should we schedule?
- What revenue should we report to investors?

**Sales Forecasting** is one of the most valuable AI applications in business.
Companies like Zalando, Amazon, and Rewe use it every single day.

Today we build a working forecasting model from scratch.

---

**Estimated Time:** 75–90 minutes  
**Dataset:** US monthly retail sales from FRED (series RSXFSN, not seasonally adjusted)  
**Libraries:** `pandas`, `numpy`, `sklearn`, `plotly`

---

> **New concept this module: Time Series**
> A time series is any data collected over time in order — stock prices, temperature readings,
> monthly sales figures. Unlike regular data, **order matters** — January always comes before February.

---
## Step 1: Install and Import Libraries

| Library | Purpose |
|---|---|
| `pandas` | Load, clean, and manipulate tabular data |
| `numpy` | Numerical operations |
| `sklearn` | Linear Regression, Pipeline, metrics |
| `plotly` | Interactive visualizations |
| `joblib` | Save and load trained models |

In [1]:
!pip install pandas numpy scikit-learn plotly --quiet
print("Libraries ready!")

Libraries ready!


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Step 2: Load Real Retail Sales Data

We use **US monthly retail sales**: official figures collected by the US Census Bureau
and distributed by the Federal Reserve Bank of St. Louis (FRED).

**Dataset:** FRED series `RSXFSN` (Advance Retail Sales: Retail Trade)
- **Coverage:** monthly, we use January 2010 to the latest published month
- **Unit:** millions of US dollars
- **Adjustment:** not seasonally adjusted

| Pattern | What it reflects |
|---|---|
| Upward trend | US retail sales grow over the years |
| December peak | Holiday shopping makes December the biggest month of the year |
| January and February drop | Spending falls sharply after the holidays |
| COVID shock | Spring 2020 shows a sudden drop, followed by a fast recovery |

> **Why "not seasonally adjusted"?**
> FRED also publishes a seasonally adjusted version of this series (`RSXFS`).
> In that version the December peak and the January drop have already been removed,
> so the line is smooth. That is useful for economists who want to see the underlying trend.
> In this module we want the model to learn the seasonal pattern itself, so we need the raw series.

The next cell downloads the data directly from FRED. No login is needed.
It also saves a copy next to the notebook, so the next run works without internet.


In [3]:
import os

# FRED series RSXFSN: US retail sales, monthly, NOT seasonally adjusted, in millions of USD
FRED_URL  = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=RSXFSN'
LOCAL_CSV = 'us_retail_sales_rsxfsn.csv'

if os.path.exists(LOCAL_CSV):
    # Use the local copy if there is one (works without internet)
    df = pd.read_csv(LOCAL_CSV, parse_dates=['date'])
    print(f"Loaded local copy: {LOCAL_CSV}")
else:
    # Otherwise download from FRED and keep a copy next to the notebook
    df = pd.read_csv(FRED_URL, names=['date', 'sales'], skiprows=1, parse_dates=['date'])
    df.to_csv(LOCAL_CSV, index=False)
    print(f"Downloaded from FRED and saved a copy as {LOCAL_CSV}")


Downloaded from FRED and saved a copy as us_retail_sales_rsxfsn.csv


In [4]:
# Keep clean numeric values from January 2010 onward
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df = df.dropna(subset=['sales'])
df = df[df['date'] >= '2010-01-01'].reset_index(drop=True)

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
print(f"\nSales statistics (USD millions):")
print(df['sales'].describe().round(2).to_string())
df.head()


Dataset shape : (200, 2)
Date range    : January 2010 to August 2026

Sales statistics (USD millions):
count       200.00
mean     455672.48
std      107137.84
min      269889.00
25%      371171.25
50%      423546.50
75%      555525.75
max      694088.00


,date,sales
0,2010-01-01,273665
1,2010-02-01,269889
2,2010-03-01,314812
3,2010-04-01,310622
4,2010-05-01,318467


---
## Step 3: Exploratory Data Analysis (EDA)

For time series, we always ask three questions first:
- Is there a **trend**? (is the line going up or down overall?)
- Is there **seasonality**? (do the same patterns repeat every year?)
- Are there **anomalies**? (unexpected spikes or crashes?)

In [5]:
# Full sales timeline
fig1 = px.line(
    df, x='date', y='sales',
    title='US Monthly Retail Sales since 2010: Can You Spot the Trend and Seasonality?',
    labels={'date': 'Month', 'sales': 'Sales (USD millions)'},
    markers=True
)
fig1.update_traces(line_color='#3498db', marker_color='#2ecc71')
fig1.update_layout(hovermode='x unified')
fig1.show()

# What to look for:
# - The overall line goes UP from left to right = upward trend
# - A sharp peak every December = seasonality (holiday shopping)
# - A drop every January and February = the quiet months after the holidays
# - A sudden dip in spring 2020 = the COVID-19 shock


In [6]:
# Add date components for analysis
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['year']       = df['date'].dt.year

# Average sales by month across all years
monthly_avg = df.groupby(['month', 'month_name'])['sales'].mean().reset_index()
monthly_avg = monthly_avg.sort_values('month')

fig2 = px.bar(
    monthly_avg,
    x='month_name', y='sales',
    title='Average Sales by Month — The Seasonal Pattern',
    labels={'sales': 'Average Sales (USD millions)', 'month_name': 'Month'},
    color='sales',
    color_continuous_scale='Blues',
    text=monthly_avg['sales'].apply(lambda x: f'{x:,.0f}')
)
fig2.update_traces(textposition='outside')
fig2.show()

# Discussion questions:
# Which month has the highest average sales? Why?
# Which months have the lowest? What business decisions follow?

In [7]:
# Year-over-year total sales: shows the growth trend
# Only complete years are compared. A year with fewer than 12 months would look like a collapse.
months_per_year = df.groupby('year')['sales'].count()
full_years = months_per_year[months_per_year == 12].index

yearly = df[df['year'].isin(full_years)].groupby('year')['sales'].sum().reset_index()
yearly['growth_pct'] = yearly['sales'].pct_change() * 100

fig3 = px.bar(
    yearly, x='year', y='sales',
    title='Total Annual Sales: Year-over-Year Growth (complete years only)',
    labels={'sales': 'Total Sales (USD millions)', 'year': 'Year'},
    color='sales',
    color_continuous_scale='Greens',
    text=yearly['sales'].apply(lambda x: f'{x:,.0f}')
)
fig3.update_traces(textposition='outside')
fig3.show()

print("Year-over-Year Growth (USD millions):")
for _, row in yearly.iterrows():
    growth_str = f"{row['growth_pct']:+.1f}%" if not pd.isna(row['growth_pct']) else "baseline"
    print(f"  {int(row['year'])}: {row['sales']:>12,.0f}  ({growth_str})")


Year-over-Year Growth (USD millions):
  2010:    3,742,372  (baseline)
  2011:    4,023,268  (+7.5%)
  2012:    4,219,819  (+4.9%)
  2013:    4,376,606  (+3.7%)
  2014:    4,555,159  (+4.1%)
  2015:    4,638,313  (+1.8%)
  2016:    4,759,153  (+2.6%)
  2017:    4,949,604  (+4.0%)
  2018:    5,172,380  (+4.5%)
  2019:    5,329,584  (+3.0%)
  2020:    5,500,685  (+3.2%)
  2021:    6,447,091  (+17.2%)
  2022:    6,974,693  (+8.2%)
  2023:    7,019,263  (+0.6%)
  2024:    7,106,697  (+1.2%)
  2025:    7,318,206  (+3.0%)


---
## Step 4: Feature Engineering

This is the most important step in time series forecasting.
A machine learning model cannot work with raw dates — we convert them
into **numerical features** that capture the patterns we saw in EDA.

We engineer two types of features:

### Type 1 — Date Features
Extract meaningful numbers from the date:

| Feature | What it captures |
|---|---|
| `month` | Seasonal patterns (month 12 = high, months 1 and 2 = low) |
| `quarter` | Q4 is always the strongest retail quarter |
| `year` | Long-term growth trend |
| `month_sin` / `month_cos` | Tells the model that January follows December (circular cycle) |
| `is_q4` | October, November, December = 1, everything else = 0 |
| `is_summer` | July, August = 1, everything else = 0 |

### Type 2 — Lag Features
Use **past sales values** to predict future ones:

| Feature | What it means |
|---|---|
| `lag_1` | Sales from 1 month ago |
| `lag_3` | Sales from 3 months ago |
| `lag_12` | Sales from exactly 1 year ago (captures annual pattern) |
| `rolling_mean_3` | Average of last 3 months (smooths out noise) |
| `rolling_mean_6` | Average of last 6 months |

> **Why lag features?**
> Last month's sales is often the best predictor of this month's sales.
> The model learns this relationship automatically from the data.

In [8]:
def engineer_time_features(df):
    """
    Engineers date and lag features from a sales DataFrame.
    Input  : DataFrame with 'date' and 'sales' columns
    Output : DataFrame with all new features added
    """
    df = df.sort_values('date').copy()

    # -- Date features ------------------------------------------
    df['month']   = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['year']    = df['date'].dt.year

    # Cyclical encoding: sin/cos so model knows Jan follows Dec
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Binary flags for key business periods
    df['is_q4']     = (df['quarter'] == 4).astype(int)
    df['is_summer'] = df['month'].isin([7, 8]).astype(int)

    # -- Lag features -------------------------------------------
    df['lag_1']  = df['sales'].shift(1)    # last month
    df['lag_3']  = df['sales'].shift(3)    # 3 months ago
    df['lag_12'] = df['sales'].shift(12)   # same month last year

    # -- Rolling statistics -------------------------------------
    df['rolling_mean_3'] = df['sales'].shift(1).rolling(window=3).mean()
    df['rolling_mean_6'] = df['sales'].shift(1).rolling(window=6).mean()
    df['rolling_std_3']  = df['sales'].shift(1).rolling(window=3).std()

    return df

# Apply to dataset
df_features = engineer_time_features(df)

# Drop rows with NaN caused by lag features at the start of the series
df_features = df_features.dropna().reset_index(drop=True)

FEATURE_COLS = ['month', 'quarter', 'year', 'month_sin', 'month_cos',
                'is_q4', 'is_summer', 'lag_1', 'lag_3', 'lag_12',
                'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3']

print(f"Rows before feature engineering : {len(df)}")
print(f"Rows after  (NaN rows dropped)  : {len(df_features)}")
print(f"\nFeatures created ({len(FEATURE_COLS)} total):")
for f in FEATURE_COLS:
    print(f"  + {f}")
print()
df_features[['date', 'sales'] + FEATURE_COLS].head(6)

Rows before feature engineering : 200
Rows after  (NaN rows dropped)  : 188

Features created (13 total):
  + month
  + quarter
  + year
  + month_sin
  + month_cos
  + is_q4
  + is_summer
  + lag_1
  + lag_3
  + lag_12
  + rolling_mean_3
  + rolling_mean_6
  + rolling_std_3



,date,sales,month,quarter,year,month_sin,month_cos,is_q4,is_summer,lag_1,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3
0,2011-01-01,293047,1,1,2011,5.000000e-01,8.660254e-01,0,0,379520.0,308709.0,273665.0,336768.333333,323713.666667,37622.439107
1,2011-02-01,293924,2,1,2011,8.660254e-01,5.000000e-01,0,0,293047.0,322076.0,269889.0,331547.666667,320122.000000,44007.718463
2,2011-03-01,338211,3,1,2011,1.000000e+00,6.123234e-17,0,0,293924.0,379520.0,314812.0,322163.666667,316447.000000,49673.977215
3,2011-04-01,332456,4,2,2011,8.660254e-01,-5.000000e-01,0,0,338211.0,293047.0,310622.0,308394.000000,322581.166667,25826.002381
4,2011-05-01,342186,5,2,2011,5.000000e-01,-8.660254e-01,0,0,332456.0,293924.0,318467.0,321530.333333,326539.000000,24080.328825
5,2011-06-01,339682,6,2,2011,1.224647e-16,-1.000000e+00,0,0,342186.0,338211.0,312635.0,337617.666667,329890.666667,4892.060745


---
## Step 5: Train/Test Split: Time Series Rules

### The Golden Rule of Time Series: Always Split by Date

| Approach | Why it's wrong or right |
|---|---|
| Random split | Future data leaks into training, so the model sees the future |
| Time-based split | Train on the past, test on the future: an honest evaluation |

We train on everything up to December 2022 and test on everything after it:

```
Jan 2011 ─────────────── Dec 2022  |  Jan 2023 ──── latest month
        TRAINING (12 years)        |   TEST (2023 onward)
                                   ^
                              cutoff point
```

The first usable month is January 2011, because the 12-month lag feature
needs one full year of history before it has a value.

> This mirrors exactly how forecasting works in real business:
> you only know the past when predicting the future.


In [9]:
TARGET  = 'sales'
CUTOFF  = '2023-01-01'

# Time-based split
train = df_features[df_features['date'] <  CUTOFF]
test  = df_features[df_features['date'] >= CUTOFF]

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET]

print(f"Training: {train['date'].min().strftime('%B %Y')} to {train['date'].max().strftime('%B %Y')} ({len(train)} months)")
print(f"Testing : {test['date'].min().strftime('%B %Y')} to {test['date'].max().strftime('%B %Y')} ({len(test)} months)")
print(f"\nThe Model Only Learned From the Past.")
print(f"No data leakage possible.")

Training: January 2011 to December 2022 (144 months)
Testing : January 2023 to August 2026 (44 months)

The Model Only Learned From the Past.
No data leakage possible.


## No Cheating — The Model Only Learned From the Past

We trained the model on data from **January 2011 to December 2022**.
We tested it on data from **January 2023 to the latest available month**.

This means:
- The model studied the **past** (2011–2022)
- We then asked it to predict the **future** (2023 onward)
- The model had **never seen** the test months before

Think of it like a university exam:
> The student studies past material.
> On exam day, they see new questions they have never seen before.
> That is the only fair way to test if they truly learned.

If we had mixed past and future data together before training,
the model would have already "seen the exam answers" —
and would score well for the wrong reason.

**The clean cutoff at January 2023 guarantees the test is fair.**

---
## Step 6: Train the Model — Linear Regression

**Linear Regression** finds the best relationship between our features and sales.
It learns a weight (coefficient) for each feature and adds them all up:

```
predicted_sales = base_value
               + (weight_lag1  × last_month_sales)
               + (weight_is_q4 × is_it_Q4)
               + (weight_year  × year)
               + ...
```

It is called Linear because the formula is always a straight weighted sum.
Simple, fast, and fully interpretable — you can see exactly what the model learned.

In [10]:
# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Model performance on the test period (January 2023 onward):")
print(f"  MAE  (Mean Absolute Error)  : USD {mae:>10,.0f} million")
print(f"  RMSE (Root Mean Sq. Error)  : USD {rmse:>10,.0f} million")
print(f"  MAPE (Mean Abs % Error)     :  {mape:>9.1f}%")
print(f"  R2   (Variance explained)   :  {r2:>9.3f}")

Model performance on the test period (January 2023 onward):
  MAE  (Mean Absolute Error)  : USD     16,064 million
  RMSE (Root Mean Sq. Error)  : USD     20,110 million
  MAPE (Mean Abs % Error)     :        2.8%
  R2   (Variance explained)   :      0.752


### Understanding Forecasting Metrics

| Metric | What it means | Target |
|---|---|---|
| **MAE** | Average prediction error in USD millions | As low as possible |
| **RMSE** | Like MAE but punishes large errors more heavily | As low as possible |
| **MAPE** | Error as % of actual sales — most business-friendly | Below 10% is good |
| **R2** | How much sales variation does the model explain? | Above 0.85 is good |

> **Discussion question:** A model with MAPE = 8% — is that good enough
> for a retailer deciding how much stock to order next month?

In [11]:
# What did the model learn?
# Coefficients show how much each feature moves the predicted sales
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

fig_coef = px.bar(
    coef_df, x='coefficient', y='feature',
    orientation='h',
    title='Feature Importance — What Drives the Sales Predictions?',
    labels={'coefficient': 'Coefficient (USD millions per unit)', 'feature': 'Feature'},
    color='coefficient',
    color_continuous_scale='RdYlGn'
)
fig_coef.show()

print("Top 5 most influential features:")
for _, row in coef_df.head(5).iterrows():
    direction = 'increases' if row['coefficient'] > 0 else 'decreases'
    print(f"  {row['feature']:<20}: {direction} prediction by USD {abs(row['coefficient']):,.0f} million per unit")

Top 5 most influential features:
  month_sin           : increases prediction by USD 27,843 million per unit
  month_cos           : decreases prediction by USD 19,884 million per unit
  is_summer           : increases prediction by USD 18,012 million per unit
  is_q4               : increases prediction by USD 13,855 million per unit
  month               : increases prediction by USD 13,659 million per unit


## Understanding: Feature Importance Chart

This chart shows the coefficient the model learned for each feature:
how much the prediction changes when that feature goes up by one unit.

### How to read it

- **Bar goes right (green):** the feature increases predicted sales
- **Bar goes left (red):** the feature decreases predicted sales
- **Longer bar:** a larger change per unit of that feature

### A trap: the bars are not directly comparable

Our features are on very different scales:

| Feature | Typical values |
|---|---|
| `is_q4`, `is_summer` | 0 or 1 |
| `month` | 1 to 12 |
| `lag_1`, `lag_12`, `rolling_mean_3` | several hundred thousand (USD millions) |

A coefficient means "change per one unit". One unit of `is_q4` is the whole difference
between the fourth quarter and the rest of the year. One unit of `lag_1` is a single
million dollars out of several hundred thousand. So the lag features show very short bars,
even though last month's sales and the same month last year are usually the strongest predictors.

To judge how much a feature really contributes, multiply its coefficient by the
typical size of that feature.

### The fix

In Step 8 we put a `StandardScaler` in front of the model. All features are then on the
same scale, and the coefficients can be compared directly.

Note what scaling does **not** do: for ordinary linear regression it does not improve the
predictions. R2 and MAPE stay practically the same. It only makes the coefficients comparable.

### One more caution

Several of our features carry almost the same information, for example `lag_1` and
`rolling_mean_3`. When features overlap like this, the model can split the effect between
them in fairly arbitrary ways. Read the coefficients as a rough guide, not as an exact ranking.

### Key takeaway

> Before you compare coefficients, check that the features are on the same scale.


---
## Step 7: Visualize Predictions vs Actual Sales

Numbers alone never tell the full story.
We build three charts:
1. Full timeline — training history + test period with predictions overlaid
2. Month-by-month comparison — actual vs predicted side by side
3. Error chart — where did the model over or under-predict?

In [12]:
pred_df = test[['date', 'sales']].copy()
pred_df['predicted'] = y_pred
pred_df['error']     = pred_df['sales'] - pred_df['predicted']
pred_df['error_pct'] = (pred_df['error'] / pred_df['sales'] * 100).round(1)

# Full timeline chart
fig_pred = go.Figure()

fig_pred.add_trace(go.Scatter(
    x=train['date'], y=train['sales'],
    name='Historical Sales (Training Data)',
    line=dict(color='#3498db', width=2),
    mode='lines+markers'
))
fig_pred.add_trace(go.Scatter(
    x=pred_df['date'], y=pred_df['sales'],
    name='Actual Sales (test period)',
    line=dict(color='#2ecc71', width=2),
    mode='lines+markers'
))
fig_pred.add_trace(go.Scatter(
    x=pred_df['date'], y=pred_df['predicted'],
    name='Model Predictions',
    line=dict(color='#e74c3c', width=2, dash='dash'),
    mode='lines+markers'
))
fig_pred.add_vline(
    x=pd.Timestamp('2023-01-01').timestamp() * 1000,
    line_dash='dot', line_color='gray', line_width=2,
    annotation_text='Train/Test cutoff', annotation_position='top right'
)
fig_pred.update_layout(
    title='Actual vs Predicted Sales — Full Timeline',
    xaxis_title='Month', yaxis_title='Sales (USD millions)',
    hovermode='x unified', height=500
)
fig_pred.show()

In [13]:
pred_df['month_name'] = pred_df['date'].dt.strftime('%b %Y')

fig_compare = go.Figure()
fig_compare.add_trace(go.Bar(
    x=pred_df['month_name'], y=pred_df['sales'],
    name='Actual', marker_color='#2ecc71'
))
fig_compare.add_trace(go.Bar(
    x=pred_df['month_name'], y=pred_df['predicted'],
    name='Predicted', marker_color='#e74c3c', opacity=0.75
))
fig_compare.update_layout(
    title='Actual vs Predicted: Month by Month (test period)',
    barmode='group',
    xaxis_title='Month', yaxis_title='Sales (USD millions)', height=450
)
fig_compare.show()

In [14]:
fig_error = px.bar(
    pred_df, x='month_name', y='error_pct',
    title='Prediction Error % by Month — Where Did the Model Struggle?',
    labels={'error_pct': 'Error (%)', 'month_name': 'Month'},
    color='error_pct',
    color_continuous_scale='RdYlGn',
    text=pred_df['error_pct'].apply(lambda x: f'{x:+.1f}%')
)
fig_error.add_hline(y=0, line_dash='dash', line_color='black')
fig_error.update_traces(textposition='outside')
fig_error.show()

# Positive = model underestimated (actual was higher than predicted)
# Negative = model overestimated (actual was lower than predicted)
print("Detailed results:")
print(pred_df[['month_name','sales','predicted','error','error_pct']].to_string(index=False))

Detailed results:
month_name  sales     predicted         error  error_pct
  Jan 2023 535871 541869.890702  -5998.890702       -1.1
  Feb 2023 516236 548310.306739 -32074.306739       -6.2
  Mar 2023 588747 600585.592709 -11838.592709       -2.0
  Apr 2023 572941 598952.511806 -26011.511806       -4.5
  May 2023 613691 609605.469054   4085.530946        0.7
  Jun 2023 594147 622810.512764 -28663.512764       -4.8
  Jul 2023 586678 615013.084765 -28335.084765       -4.8
  Aug 2023 607777 611840.886573  -4063.886573       -0.7
  Sep 2023 572898 590324.520436 -17426.520436       -3.0
  Oct 2023 586400 596506.360802 -10106.360802       -1.7
  Nov 2023 598630 621819.632177 -23189.632177       -3.9
  Dec 2023 645247 655000.130803  -9753.130803       -1.5
  Jan 2024 529976 552500.034302 -22524.034302       -4.3
  Feb 2024 534513 556149.467311 -21636.467311       -4.0
  Mar 2024 586713 607680.657275 -20967.657275       -3.6
  Apr 2024 582673 588098.841364  -5425.841364       -0.9
  May 2024 61

---
## Step 8: AI Workflow Integration: sklearn Pipeline

We package the full workflow into a single deployable Pipeline:

```
Raw Features → StandardScaler → LinearRegression → Sales Prediction
```

`StandardScaler` puts every feature on the same scale (mean 0, standard deviation 1).
For ordinary linear regression this does not improve the predictions: you should see
practically the same R2 and MAPE as in Step 6. (A difference in the third decimal can appear,
because values such as 2023 next to values such as 0 and 1 are numerically awkward for the solver.)
What changes is that the coefficients become comparable, which is what we needed for the
feature importance chart.

Many other models do need scaled inputs (regularised regression, neural networks,
k-nearest neighbours), so building the scaler into the pipeline is good practice.


In [15]:
# Build pipeline
forecast_pipeline = Pipeline([
    ('scaler', StandardScaler()),     # put all features on the same scale
    ('model',  LinearRegression())    # learn the sales pattern
])

# Train in one call
forecast_pipeline.fit(X_train, y_train)

# Evaluate
y_pred_pl  = forecast_pipeline.predict(X_test)
pl_r2      = r2_score(y_test, y_pred_pl)
pl_mape    = np.mean(np.abs((y_test - y_pred_pl) / y_test)) * 100

print(f"Pipeline R2   : {pl_r2:.3f}")
print(f"Pipeline MAPE : {pl_mape:.1f}%")
print()

# Coefficients on scaled features can be compared directly
scaled_coef = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': forecast_pipeline.named_steps['model'].coef_
}).sort_values('coefficient', key=abs, ascending=False)
print("Coefficients on scaled features (largest first):")
print(scaled_coef.to_string(index=False, float_format=lambda v: f"{v:,.0f}"))
print()

# Save the trained pipeline so it can be reused without retraining
joblib.dump(forecast_pipeline, 'sales_forecast_pipeline.pkl')
print("Pipeline saved to: sales_forecast_pipeline.pkl")


Pipeline R2   : 0.752
Pipeline MAPE : 2.8%

Coefficients on scaled features (largest first):
       feature  coefficient
         month       47,153
         lag_1       27,950
     month_sin       19,688
        lag_12       17,021
 rolling_std_3       15,801
rolling_mean_3       14,289
     month_cos      -14,060
rolling_mean_6       12,979
       quarter      -11,741
     is_summer        6,713
         lag_3        6,441
         is_q4        5,999
          year       -5,554

Pipeline saved to: sales_forecast_pipeline.pkl


In [16]:
# Forecast the 3 months after the last month in the data
forecasts = []
last_sales = list(y_test.values)  # keep track of predicted values

for i in range(3):
    next_month = df['date'].max() + pd.DateOffset(months=i+1)

    row = pd.DataFrame([{
        'month'         : next_month.month,
        'quarter'       : next_month.quarter,
        'year'          : next_month.year,
        'month_sin'     : np.sin(2 * np.pi * next_month.month / 12),
        'month_cos'     : np.cos(2 * np.pi * next_month.month / 12),
        'is_q4'         : int(next_month.month in [10, 11, 12]),
        'is_summer'     : int(next_month.month in [7, 8]),
        'lag_1'         : last_sales[-1],
        'lag_3'         : last_sales[-3],
        'lag_12'        : last_sales[-12],
        'rolling_mean_3': np.mean(last_sales[-3:]),
        'rolling_mean_6': np.mean(last_sales[-6:]),
        'rolling_std_3' : np.std(last_sales[-3:], ddof=1)
    }])

    pred = forecast_pipeline.predict(row)[0]
    last_sales.append(pred)  # use prediction as input for next month
    forecasts.append({'month': next_month.strftime('%B %Y'), 'forecast': pred})
    print(f"Forecast for {next_month.strftime('%B %Y')} : USD {pred:,.0f} million")

Forecast for September 2026 : USD 630,149 million
Forecast for October 2026 : USD 648,171 million
Forecast for November 2026 : USD 668,863 million


---
## YOUR TURN — Exercises

---

### Exercise 1 — Interpret the Feature Importance Chart

Look at the Feature Importance chart from Step 6 and answer:

1. Which feature has the highest impact on predictions?
2. Does `is_q4` increase or decrease predicted sales? By how much?
3. Does `is_summer` make business sense? Why?

---

### Exercise 2 — Add a New Feature

Add `is_december` (1 = December, 0 = everything else) to `engineer_time_features()`.
Retrain. Does accuracy improve?

> Hint: `df['is_december'] = (df['month'] == 12).astype(int)`

---

### Exercise 3 — Forecast the Next 6 Months

Using the saved pipeline, predict sales for the 6 months after the last month in the data.
Create a plotly bar chart showing the 6-month forecast.

> This is exactly the output a business manager would want

---

## Key Takeaways

| Concept | What You Learned |
|---|---|
| **Time Series** | Data ordered in time — order matters, patterns repeat |
| **Date Features** | Extract month, quarter, year from raw dates |
| **Lag Features** | Use past values to predict future ones |
| **Cyclical Encoding** | sin/cos tells the model months form a yearly cycle |
| **Time-based Split** | Never randomly split time series — always split by date |
| **Linear Regression** | Learns a weight per feature, adds them up |
| **MAPE** | Most business-friendly metric — error as % of actual |
| **Pipeline** | Packages scaler + model into one deployable object |

---